# Practical PyTorch · Part 14 — Companion Notebook

This goes with **"Three Lines and a Verdict: Running Models with Hugging Face Pipelines"**. Run the cells top to bottom (Shift+Enter).

The whole idea: name a *task*, optionally name a *model*, hand it an input, read the output. Three lines, any modality.

For speed on the heavier tasks: **Runtime → Change runtime type → Hardware accelerator → GPU**.

## 0. Install transformers

Colab ships with PyTorch, but not `transformers`. Install it once per session (this is quick).

In [ ]:
!pip install -q transformers

## 1. The famous three lines

If these look familiar, they should — you ran them as the "easy button" warm-up in Part 9. This is the promised explanation. Sentiment analysis is the classic first pipeline. The first call downloads a default model, so give it a few seconds; later calls are fast.

In [ ]:
from transformers import pipeline

clf = pipeline("sentiment-analysis")
print(clf("I can't believe how easy this was."))
print(clf("Setting up a GPU used to ruin my whole afternoon."))

Notice the output: a **list of dicts**, even for one input. Reach for `result[0]['label']`, not `result['label']`.

In [ ]:
result = clf("This is the third best day of my life.")
print(result)
print("label:", result[0]["label"], "| score:", round(result[0]["score"], 4))

## 2. Summarization

Feed it a wall of text, get back a short version. `max_length` / `min_length` are token budgets for the summary — rough dials, not exact character counts. The default summarization model is on the chunky side, so the download takes a moment.

In [ ]:
long_article = (
    "PyTorch is an open-source machine learning library used to build and run "
    "neural networks. It was originally developed by Meta AI and is now part of "
    "the Linux Foundation. Researchers favor it for its flexible, Pythonic style, "
    "while companies rely on it to ship models into production. A huge share of "
    "the open models you read about — language models, image generators, speech "
    "recognizers — are built with PyTorch and distributed as PyTorch weights, "
    "which is why learning to drive it opens up so much of the modern AI world."
)

summarize = pipeline("summarization")
print(summarize(long_article, max_length=60, min_length=20))

## 3. Zero-shot classification

Sort text into categories you invent **on the spot** — no training. You pass the labels in `candidate_labels`, and the model scores how well the text fits each one.

In [ ]:
route = pipeline("zero-shot-classification")
out = route(
    "My card was charged twice this month.",
    candidate_labels=["billing", "tech support", "sales"],
)
print(out["labels"][0], "->", round(out["scores"][0], 4))
print(out)

Zero-shot is the oddball output: a **single dict** with parallel `labels` and `scores` lists, sorted best-first. Swap in your own labels and text and try routing a few support messages.

## 4. Image classification

Same three-line shape, different modality. Pass a URL, a local file path, or a PIL image — the pipeline figures out how to load it.

In [ ]:
vision = pipeline("image-classification")
img_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png"
for guess in vision(img_url)[:3]:
    print(round(guess["score"], 3), guess["label"])

## 5. Automatic speech recognition (mention)

Transcribing audio works the same way — `pipeline("automatic-speech-recognition")` taking an audio file path. It needs an audio backend, so it's the one task that may want an extra install. Here's the shape; uncomment and point it at an audio file to try it.

```python
# !pip install -q librosa
# asr = pipeline("automatic-speech-recognition")
# print(asr("some_audio.wav"))
```

## 6. Picking your own model

The defaults are fine for kicking the tires. To use a *specific* model, name it with `model=` — that string is its address on the Hugging Face Hub (`org/name`). Any model you find on the Hub drops into this slot, as long as it matches the task.

In [ ]:
clf = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)
print(clf("Naming the model myself feels like grown-up PyTorch."))

## 7. Use the GPU

By default a pipeline runs on the CPU. With a GPU attached, put the model on it with `device=0` ("the first GPU"). The guard below uses the GPU when there is one and falls back to CPU (`-1`) when there isn't, so this cell runs either way.

In [ ]:
import torch

device = 0 if torch.cuda.is_available() else -1
print("using", "GPU" if device == 0 else "CPU")

clf = pipeline("sentiment-analysis", device=device)
print(clf("On the GPU this is the same call, just faster."))

That's the fast path: name a task, optionally name a model, hand it an input, read the output — across text, images, and audio. Next up, **Part 15 — Beyond Pipelines**, where we open the box for batching, embeddings, and more control.